# Install required libraries
!pip install pymupdf4llm pymupdf chromadb sentence-transformers pillow pytesseract transformers torch

## 1- Setup and PDF Discovery

In [1]:
import os
import re
from pathlib import Path

# Define the homework folder path
HOMEWORK_FOLDER = "./hw_ds442"  # Update this path to your homework folder

# Discover all homework PDF files
pdf_files = sorted([f for f in os.listdir(HOMEWORK_FOLDER) if re.match(r'hw\d+\.pdf', f)])

print(f"Found {len(pdf_files)} homework files:")
for pdf in pdf_files:
    file_path = os.path.join(HOMEWORK_FOLDER, pdf)
    file_size = os.path.getsize(file_path) / (1024 * 1024)  # Size in MB
    print(f"  - {pdf} ({file_size:.2f} MB)")

Found 7 homework files:
  - hw1.pdf (0.42 MB)
  - hw2.pdf (0.50 MB)
  - hw3.pdf (0.46 MB)
  - hw4.pdf (0.54 MB)
  - hw5.pdf (0.44 MB)
  - hw6.pdf (0.63 MB)
  - hw7.pdf (0.30 MB)


## 2- Image Captioning Model Setup

In [2]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import torch

# Load BLIP-base once (reuse for multiple images)
processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base", use_safetensors=True, dtype=torch.float32
)
model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base", use_safetensors=True, dtype=torch.float32
)
device = "cpu"
model.to(device)


def generate_img_caption(image: Image.Image) -> str:
    # Ensure RGB
    image = image.convert("RGB")

    # Resize to BLIP-base default
    image = image.resize((384, 384))

    # Convert to PyTorch tensor manually (no NumPy)
    img_tensor = torch.tensor(list(image.getdata()), dtype=torch.float32) / 255.0
    img_tensor = img_tensor.view(384, 384, 3)              # H, W, C
    img_tensor = img_tensor.permute(2, 0, 1).unsqueeze(0)  # B, C, H, W

    # Move to device
    inputs = {"pixel_values": img_tensor.to(device)}

    # Generate caption
    output_ids = model.generate(**inputs)
    caption = processor.decode(output_ids[0], skip_special_tokens=True)

    return caption


# ------------------ Test the function ------------------
if __name__ == "__main__":
    test_image_path = "./hw_ds442/hw1-Q1_img1.png"
    test_image = Image.open(test_image_path)
    caption = generate_img_caption(test_image)
    print(f"Generated caption for {test_image_path}:")
    print(caption)


/Users/bonam/Documents/GitHub/ALAASKA_RAG/aa_rag_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Generated caption for ./hw_ds442/hw1-Q1_img1.png:
a diagram of the three phases of the algorithm


## 3- PDF Text and Image Extraction
### With Question Pattern Recognition and image captionning

In [21]:
import os
import re
import fitz
from typing import List, Dict, Any

OUTPUT_FOLDER = "./hw_ds442_extract/hw7"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

def parse_questions_from_pdf(pdf_path: str, hw_id: str) -> List[Dict[str, Any]]:
    doc = fitz.open(pdf_path)
    questions = []

    # Question pattern: Matches Q1, Q 1, Q1.1, Q 1.1, etc. at start of line
    question_pattern = re.compile(r'(?<!Answer\s)(?<!Answer)(?:^|\s)Q\s*(\d+(?:\.\d+)*)\b', re.I)
    # Answer pattern: Matches "Answer Q1.1" or "Answer Q 1.1" with optional space
    answer_pattern = re.compile(r'^Answer\s+Q\s*(\d+(?:\.\d+)*)', re.I)
    explanation_pattern = re.compile(r'^Explanation[：:]', re.I)

    # Flatten PDF text per line with page number
    all_lines = []
    for page_num, page in enumerate(doc, start=1):
        for line in page.get_text().split('\n'):
            all_lines.append((line.strip(), page_num))

    i = 0
    while i < len(all_lines):
        line, page_num = all_lines[i]
        q_match = question_pattern.search(line)
        if q_match:
            q_id = q_match.group(1)
            current_question = {
                'hw_id': hw_id,
                'question_id': f"Q{q_id}",
                'chunk_id': f"{hw_id}-Q{q_id}",
                'parent_question': '',
                'question_text': '',
                'answer_text': '',
                'explanation_text': '',
                'images': [],
                'page_num': page_num
            }

            # Parent context if parent exists
            if '.' in q_id:
                parent_id = q_id.rsplit('.', 1)[0]
                parent = next((q for q in questions if q['question_id'] == f"Q{parent_id}"), None)
                if parent:
                    current_question['parent_question'] = parent['question_text']

            # --- Collect Question Text ---
            q_lines = [line]
            i += 1
            while i < len(all_lines):
                next_line, _ = all_lines[i]
                if question_pattern.search(next_line) or answer_pattern.search(next_line):
                    break
                q_lines.append(next_line)
                i += 1
            current_question['question_text'] = '\n'.join(q_lines).strip()

            # --- Collect Answer Text ---
            a_lines = []
            while i < len(all_lines):
                next_line, _ = all_lines[i]
                ans_match = answer_pattern.search(next_line)
                if ans_match and ans_match.group(1) == q_id:
                    a_lines.append(next_line)
                    i += 1
                    while i < len(all_lines):
                        l, _ = all_lines[i]
                        if explanation_pattern.search(l) or question_pattern.search(l):
                            break
                        a_lines.append(l)
                        i += 1
                    break
                else:
                    break  # stop if no answer matches current question
            current_question['answer_text'] = '\n'.join(a_lines).strip()

            # --- Collect Explanation Text ---
            e_lines = []
            while i < len(all_lines):
                next_line, _ = all_lines[i]
                if explanation_pattern.search(next_line):
                    e_lines.append(next_line)
                    i += 1
                    while i < len(all_lines):
                        l, _ = all_lines[i]
                        if question_pattern.search(l):
                            break
                        e_lines.append(l)
                        i += 1
                    break
                else:
                    break
            current_question['explanation_text'] = '\n'.join(e_lines).strip()

            # --- Extract images from the page and add captions ---
            for img_index, img in enumerate(doc[page_num-1].get_images(full=True)):
                xref = img[0]
                base_image = doc.extract_image(xref)
                img_bytes = base_image["image"]
                img_ext = base_image["ext"]
                img_filename = f"{current_question['chunk_id']}_img{img_index+1}.{img_ext}"
                img_path = os.path.join(OUTPUT_FOLDER, img_filename)
                with open(img_path, "wb") as f:
                    f.write(img_bytes)

                # --- NEW: generate caption ---
                pil_image = Image.open(img_path)
                img_caption = generate_img_caption(pil_image)

                current_question['images'].append({
                    'caption': img_caption,  # <-- added caption
                    'page': page_num,
                    'size': base_image['width'],
                    'filename': img_filename
                })

            questions.append(current_question)
        else:
            i += 1

    return questions


def save_question_as_txt(question: Dict[str, Any], folder: str = OUTPUT_FOLDER):
    file_path = os.path.join(folder, f"{question['chunk_id']}.txt")
    with open(file_path, 'w', encoding='utf-8') as f:
        f.write(f"Homework ID: {question['hw_id']}\n")
        f.write(f"Question ID: {question['question_id']}\n")
        f.write(f"Chunk ID: {question['chunk_id']}\n")
        f.write(f"Parent Question: {question['parent_question']}\n\n")
        f.write(f"Question Text:\n{question['question_text']}\n\n")
        f.write(f"Answer Text:\n{question['answer_text']}\n\n")
        f.write(f"Explanation Text:\n{question['explanation_text']}\n\n")
        f.write(f"Images:\n")
        for img in question['images']:
            f.write(f"  - Image Caption: {img['caption']}, Filename: {img['filename']}, Size: {img['size']}, Page {img['page']} \n")
    print(f"Saved question {question['chunk_id']} to {file_path}")


# --- Example Usage --- 
pdf_files = [f for f in os.listdir(HOMEWORK_FOLDER) if f.lower().endswith('.pdf')]
current_pdf = next(f for f in pdf_files if f.lower() == "hw7.pdf")
if pdf_files: 
    test_file = os.path.join(HOMEWORK_FOLDER, current_pdf) 
    hw_id = os.path.splitext(current_pdf)[0] 

    # Parse 
    questions = parse_questions_from_pdf(test_file, hw_id) 
    
    # Save each question as txt 
    for q in questions: 
        save_question_as_txt(q) 
    print(f"\nParsed and saved {len(questions)} questions from {current_pdf}") 
else: print("No PDF files found.")

Saved question hw7-Q1 to ./hw_ds442_extract/hw7/hw7-Q1.txt
Saved question hw7-Q1.1 to ./hw_ds442_extract/hw7/hw7-Q1.1.txt
Saved question hw7-Q1.2 to ./hw_ds442_extract/hw7/hw7-Q1.2.txt
Saved question hw7-Q2 to ./hw_ds442_extract/hw7/hw7-Q2.txt
Saved question hw7-Q3 to ./hw_ds442_extract/hw7/hw7-Q3.txt
Saved question hw7-Q4 to ./hw_ds442_extract/hw7/hw7-Q4.txt
Saved question hw7-Q5 to ./hw_ds442_extract/hw7/hw7-Q5.txt
Saved question hw7-Q5.1 to ./hw_ds442_extract/hw7/hw7-Q5.1.txt
Saved question hw7-Q5.2 to ./hw_ds442_extract/hw7/hw7-Q5.2.txt
Saved question hw7-Q5.2.1 to ./hw_ds442_extract/hw7/hw7-Q5.2.1.txt
Saved question hw7-Q5.2.2 to ./hw_ds442_extract/hw7/hw7-Q5.2.2.txt
Saved question hw7-Q5.2.3 to ./hw_ds442_extract/hw7/hw7-Q5.2.3.txt
Saved question hw7-Q5.2.4 to ./hw_ds442_extract/hw7/hw7-Q5.2.4.txt
Saved question hw7-Q6 to ./hw_ds442_extract/hw7/hw7-Q6.txt

Parsed and saved 14 questions from hw7.pdf


## 4- Build ChromaDB Collection
Embed question + parent context + image captions as primary chunk Embed answer only in metadata, not in text (so retrieval can access it but LLM cannot hallucinate it)

In [87]:
#import chromadb
#chromadb.api.client.SharedSystemClient.clear_system_cache()

In [22]:
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

# --- Initialize embedding model ---
embedding_model = SentenceTransformer('BAAI/bge-base-en-v1.5')

# --- Initialize new ChromaDB client ---
client = chromadb.PersistentClient(path="./ds442_vdb", settings=Settings(allow_reset=True))

# --- Create or get collection ---
collection_name = "ds442_hw_sol"
existing_collections = [c.name for c in client.list_collections()]
if collection_name in existing_collections:
    collection = client.get_collection(collection_name)
else:
    collection = client.create_collection(name=collection_name)

# --- Function to ingest questions ---
def ingest_questions_to_chroma(questions):
    for q in questions:
        # Construct chunk text for embedding
        image_captions = "; ".join(img.get('caption', '') for img in q['images'])
        chunk_text = "\n".join(filter(None, [
            q.get('parent_question', ''), 
            q['question_text'], 
            f"Image captions: {image_captions}" if image_captions else ""
        ]))

        # Metadata (simple types only)
        metadata = {
            'hw_id': q['hw_id'],
            'question_id': q['question_id'],
            'chunk_id': q['chunk_id'],
            'page_num': q['page_num'],
            'answer_text': q['answer_text'],
            'image_captions': image_captions  # <-- only string of captions
        }

        # Compute embedding as PyTorch tensor (avoids NumPy)
        embedding = embedding_model.encode(
            chunk_text,
            normalize_embeddings=True,
            convert_to_numpy=False  # returns PyTorch tensor
        )

        # Upsert into ChromaDB
        collection.upsert(
            ids=[q['chunk_id']],
            embeddings=[embedding.cpu().tolist()],
            metadatas=[metadata],
            documents=[chunk_text]
        )
    print(f"Upserted {len(questions)} questions into ChromaDB collection '{collection_name}'")

# --- Example usage ---
ingest_questions_to_chroma(questions)


Upserted 14 questions into ChromaDB collection 'ds442_hw_sol'


## 5- Test the Chroma Db with a simple query

In [4]:
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

# Initialize embedding model (same as ingestion)
embedding_model = SentenceTransformer('BAAI/bge-base-en-v1.5')

# Connect to existing ChromaDB
client = chromadb.PersistentClient(path="./ds442_vdb", settings=Settings(allow_reset=True))
collection = client.get_collection(name="ds442_hw_sol")

# Query function
def query_chroma(query_text, n_results):
    # Generate embedding for query
    query_embedding = embedding_model.encode(
        query_text,
        normalize_embeddings=True,
        convert_to_numpy=False
    )
    
    # Search in ChromaDB
    results = collection.query(
        query_embeddings=[query_embedding.cpu().tolist()],
        n_results=n_results
    )
    
    # Extract and display results
    for i in range(len(results['ids'][0])):
        print(f"Rank #{i+1}")
        print(f"Chunk ID: {results['metadatas'][0][i]['chunk_id']}")
        print(f"Question: {results['documents'][0][i]}")
        print(f"Answer: {results['metadatas'][0][i]['answer_text']}")
        print(f"Distance: {results['distances'][0][i]}")
        print("-" * 60) ##
    
    return results

# Example query
query = "Consider the gridworld where Left and Right actions are successful 100% of the time. Specifically, the available actions in each state are to move to the neighboring grid squares. From state a, there is also an exit action available, which results in going to the terminal state and collecting a reward of 10. Similarly, in state e, the reward for the exit action is 1. Exit actions are successful 100% of the time. The discount factor (γ) is 0.9. Q7.1 We will execute one round of policy iteration. Consider the policy π_i shown below and evaluate the following quantities for this policy. V^(π_i ) (a)=[] V^(π_i ) (b)=[] V^(π_i ) (c)=[] V^(π_i ) (d)=[] V^(π_i ) (e)=[____] "
results = query_chroma(query, n_results=3)

Rank #1
Chunk ID: hw4-Q7
Question: Q7 Policy Iteration
Points: 5
Consider the gridworld where Left and Right actions are successful 100% of the time. Specifically, the
available actions in each state are to move to the neighboring grid squares. From state 𝑎, there is also an
exit action available, which results in going to the terminal state and collecting a reward of 10. Similarly,
in state 𝑒, the reward for the exit action is 1. Exit actions are successful 100% of the time.
The discount factor (γ) is 0.9.
Image captions: a gray color with a white background
Answer: 
Distance: 0.06972595304250717
------------------------------------------------------------
Rank #2
Chunk ID: hw4-Q6
Question: Q6 Policy Evaluation
Points: 10
Consider the gridworld where Left and Right actions are successful 100% of the time. Specifically, the
available actions in each state are to move to the neighboring grid squares. From state 𝑎, there is also
an exit action available, which results in going to the ter